# 03 — Fairness Audit

Responsible-AI check across protected attributes (sex, education, age group).
Detect-and-disclose only — mirrors `src/fairness.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
pd.set_option('display.max_columns', 40)

In [ ]:
import joblib
from src.preprocessing import split_data
from src import fairness
bundle = joblib.load(config.MODEL_PATH)
df = pd.read_csv(config.PROCESSED_CSV)
_, X_test, _, y_test = split_data(df)

## Per-group metrics

In [ ]:
audit = fairness.run_fairness_audit(bundle['model'], X_test, y_test)
audit

## Disparity summary

In [ ]:
fairness.disparity_summary(audit)

## Selection rate by group

In [ ]:
for attr in audit['attribute'].unique():
    sub = audit[audit['attribute']==attr]
    plt.figure(figsize=(5,2.5))
    plt.bar(sub['group'], sub['predicted_positive_rate'])
    plt.title(f'Predicted high-risk rate by {attr}'); plt.xticks(rotation=30); plt.show()

**Interpretation:** groups whose selection-rate ratio falls below ~0.8 (the
'80% rule') are disclosed as fairness gaps. Age group shows the largest
disparity here. See `reports/fairness_audit_report.md` for the written report.